In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Supervised Classification Framework for Squat Form Quality Assessment

## 1. Predictive Modeling Architecture
This phase of the project transitions from kinematic feature extraction to supervised pattern recognition. The objective is to construct a robust mathematical classifier capable of mapping a high-dimensional feature vector $\mathbf{x} \in \mathbb{R}^d$ (representing a single completed squat repetition) to a binary class label $y \in \{0, 1\}$, where:

$$y = \begin{cases} 1, & \text{Good Form} \\ 0, & \text{Bad Form} \end{cases}$$

We deploy a **Support Vector Machine (SVM)** classifier, which is structurally optimized for high-dimensional, small-sample datasets typical of specialized biomechanical tracking applications.

## 2. Feature Selection & Dimensionality Engineering
The dataset contains mixed kinematic dimensions, including temporal durations, spatial drift coefficients, and angular extrema. 

To ensure the classification model targets cross-subject structural posture rather than individual execution speeds or anthropometric variances, we drop specific non-geometric features via `COLUMNS_TO_DROP`.

### 2.1 Theoretical Justification for Dropping Features
1. **Temporal Features (`_duration_seconds`):** A rep executed slowly by an experienced lifter or rapidly by a beginner could confound a classifier if duration is treated as an indicative marker of form correctness.
2. **Kinetic Velocities (`_velocity_y`):** Absolute pixel displacement velocity varies directly based on camera distance, resolution, and frame rate, introducing non-generalizable variance.
3. **Data Leakage Preventatives (`video_name`, `rep_number`):** Identification metadata must be isolated to prevent the learning algorithm from over-fitting to specific background configurations or subject identities.

In [ ]:
DATA_PATH = "../edited_datasets/squat_rep_vectors_ml.csv"
OUTPUT_PATH = "../models/"

COLUMNS_TO_DROP = [
    "video_name",
    "rep_number",
    "label",
    "body_side",
    "descent_duration_seconds",
    "ascent_duration_seconds",
    "bottom_pause_duration_seconds",
    "descent_to_ascent_ratio",
    "total_rep_duration",
    "time_to_max_jerk",
    "avg_descent_velocity_y",
    "avg_ascent_velocity_y",
    "max_acceleration_y",
    "max_jerk_y",
    "peak_ascent_velocity_y",
    "sticking_point_velocity_drop",
    "velocity_loss_vs_rep_1",
]

In [ ]:
def load_dataset(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {path}")

    return pd.read_csv(path)

In [ ]:
def prepare_features(df):
    y = df["label"].map({"good": 1, "bad": 0}).values
    groups = df["video_name"].values

    X_raw = df.drop(columns=COLUMNS_TO_DROP)
    feature_names = X_raw.columns.tolist()

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)

    return X, y, groups, feature_names, scaler

In [ ]:
def create_model():
    return SVC(
        kernel="linear",
        C=1.0,
        probability=True,
        class_weight="balanced",
        random_state=42,
    )

## 3. Leakage-Proof Validation via GroupKFold Cross-Validation
Evaluating a computer-vision-based classifier using traditional random cross-validation yields highly optimistic, fraudulent performance metrics due to **subject-level data leakage**. Multiple repetitions from the same individual video share identical anthropometric lengths, camera angles, and clothing backgrounds.

To test true algorithmic generalization to unseen individuals, we implement a **GroupKFold** cross-validation strategy where groups are explicitly clustered by the `video_name` parameter.

### 3.1 Group Segmentation Mathematics
Let the set of all unique video files be partitioned into $K$ mutually exclusive subsets $\mathcal{G}_1, \mathcal{G}_2, \dots, \mathcal{G}_K$ such that:

$$\bigcup_{k=1}^K \mathcal{G}_k = \mathcal{G} \quad \text{and} \quad \mathcal{G}_a \cap \mathcal{G}_b = \emptyset \quad \forall a \neq b$$

During the $k$-th iteration, all repetitions originating from videos belonging to group $\mathcal{G}_k$ are strictly sequestered to form the test partition, while the remaining $K-1$ groups constitute the training matrix. This ensures that the accuracy reported reflects real-world deployment metrics on entirely novel users.

In [ ]:
def cross_validate(model, X, y, groups, threshold=0.35):
    gkf = GroupKFold(n_splits=3)

    oof_preds = np.zeros(len(y))
    oof_probs = np.zeros(len(y))

    print("\n--- Starting Leak-Proof Cross Validation ---")

    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1
    ):
        fold_model = create_model()

        fold_model.fit(X[train_idx], y[train_idx])

        probs = fold_model.predict_proba(X[val_idx])[:, 1]
        preds = np.where(probs < threshold, 0, 1)

        oof_probs[val_idx] = probs
        oof_preds[val_idx] = preds

        acc = np.mean(preds == y[val_idx])
        print(f"Fold {fold}: {acc:.2f}")

    return oof_preds, oof_probs

In [ ]:
def print_metrics(y_true, preds, probs):
    print("\n=== OVERALL PERFORMANCE ===")
    print(classification_report(y_true, preds, target_names=["bad", "good"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, preds))
    print(f"ROC-AUC: {roc_auc_score(y_true, probs):.4f}")

In [ ]:
def train_model(X, y):
    model = create_model()
    model.fit(X, y)
    return model

In [ ]:
def print_feature_importance(model, feature_names, top_n=5):
    importances = np.abs(model.coef_[0])
    indices = np.argsort(importances)[::-1]

    print("\nTop Features:")
    for i in range(min(top_n, len(indices))):
        idx = indices[i]
        print(
            f"{i+1}. {feature_names[idx]} "
            f"(Weight: {importances[idx]:.4f})"
        )

In [ ]:
def save_artifacts(model, scaler, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    joblib.dump(model, os.path.join(output_dir, "squat_classifier_model.pkl"))
    joblib.dump(scaler, os.path.join(output_dir, "squat_scaler.pkl"))

    print(f"\nSaved model and scaler to '{output_dir}'")

In [ ]:
def create_squad_model():
    df = load_dataset(DATA_PATH)

    X, y, groups, feature_names, scaler = prepare_features(df)

    print(f"Dataset Loaded: {X.shape[0]} repetitions, {X.shape[1]} features.")

    preds, probs = cross_validate(create_model(), X, y, groups)

    print_metrics(y, preds, probs)

    print("\n--- Training Production Model ---")
    model = train_model(X, y)

    print_feature_importance(model, feature_names)

    save_artifacts(model, scaler, OUTPUT_PATH)

In [3]:
create_squad_model()

Dataset Loaded: 35 repetitions, 18 features.

--- Starting Leak-Proof Cross Validation ---
Fold 1: 1.00
Fold 2: 0.33
Fold 3: 0.83

=== OVERALL PERFORMANCE ===
              precision    recall  f1-score   support

         bad       0.64      0.64      0.64        14
        good       0.76      0.76      0.76        21

    accuracy                           0.71        35
   macro avg       0.70      0.70      0.70        35
weighted avg       0.71      0.71      0.71        35

Confusion Matrix:
[[ 9  5]
 [ 5 16]]
ROC-AUC: 0.8231

--- Training Production Model ---

Top Features:
1. max_torso_lean_degrees (Weight: 0.5791)
2. torso_lean_to_depth_ratio (Weight: 0.5095)
3. max_depth_ratio (Weight: 0.4952)
4. min_knee_angle (Weight: 0.4809)
5. knee_x_sd (Weight: 0.3812)

Saved model and scaler to '../models/'


c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The

## 4. Data Normalization & Maximum-Margin Classification

### 4.1 Standard Scaling (Z-Score Normalization)
Support Vector Machines are highly sensitive to the absolute scale of input features due to their reliance on Euclidean distance metrics. To equalize feature contributions, we apply a linear transformation converting each feature vector element to a zero-mean, unit-variance distribution:

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

Where $\mu_j$ is the mean of feature $j$, and $\sigma_j$ is its standard deviation:

$$\mu_j = \frac{1}{N}\sum_{i=1}^{N} x_{ij}, \quad \sigma_j = \sqrt{\frac{1}{N}\sum_{i=1}^{N} (x_{ij} - \mu_j)^2}$$

### 4.2 Support Vector Machine (SVM) Formulation
Given our normalized training dataset $D = \{(\mathbf{z}_i, y_i)\}_{i=1}^N$, the soft-margin linear SVM constructs an optimal separating hyperplane $\mathbf{w}^T \mathbf{z} + b = 0$ by solving the following constrained primal optimization problem:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^{N} \xi_i$$

$$\text{subject to } y_i (\mathbf{w}^T \mathbf{z}_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0 \quad \forall i$$

Where:
* $\mathbf{w}$ represents the weight vector normal to the separating hyperplane.
* $C > 0$ is the regularization parameter controlling the trade-off between maximizing the margin geometric width ($2 / \|\mathbf{w}\|$) and minimizing training classification errors.
* $\xi_i$ are slack variables allowing bound violations for non-linearly separable instances.

In [7]:
def analyze_new_repetition(rep_vector_dict):
    """
    Receives a single repetition's 31 features as a dictionary,
    filters the active 18 features, scales them, and provides diagnostics.
    """
    # Load your trained model and scaler
    model = joblib.load("../models/squat_classifier_model.pkl")
    scaler = joblib.load("../models/squat_scaler.pkl")
    
    # Convert input to DataFrame matching the original structure
    rep_df = pd.DataFrame([rep_vector_dict])
    
    # Define columns to drop exactly as done in training
    COLUMNS_TO_DROP = [
        "video_name", "rep_number", "label", "body_side",
        "descent_duration_seconds", "ascent_duration_seconds",
        "bottom_pause_duration_seconds", "descent_to_ascent_ratio",
        "total_rep_duration", "time_to_max_jerk",
        "avg_descent_velocity_y", "avg_ascent_velocity_y",
        "max_acceleration_y", "max_jerk_y", "peak_ascent_velocity_y",
        "sticking_point_velocity_drop", "velocity_loss_vs_rep_1"
    ]
    
    # Drop non-feature elements if they exist in the input payload
    cols_to_drop = [col for col in COLUMNS_TO_DROP if col in rep_df.columns]
    X_raw = rep_df.drop(columns=cols_to_drop)
    
    # Scale features using your fitted production scaler
    X_scaled = scaler.transform(X_raw)
    
    # Run ML prediction (1 = good, 0 = bad)
    prediction = model.predict(X_scaled)[0]
    
    feedback = []
    if prediction == 1:
        status = "Good Form"
        feedback.append("Great execution! Keep maintaining this depth and torso alignment.")
    else:
        status = "Bad Form Detected"
        
        # Heuristic Diagnostics (using raw dictionary metrics)
        if rep_vector_dict.get('max_depth_ratio', 1.0) < 0.25:
            feedback.append("• Problem: Shallow depth. Ensure your hips drop lower (hip crease below parallel with knees).")
            
        if rep_vector_dict.get('max_torso_lean_degrees', 0) > 60.0:
            feedback.append("• Problem: Excessive torso lean. Your chest is dropping forward too much, putting strain on your lower back.")
            
        if rep_vector_dict.get('sticking_point_velocity_drop', 0) > 0.003:
            feedback.append("• Problem: Severe velocity drop on ascent. Work on explosive power out of the bottom position.")
            
        if not feedback:
            feedback.append("• Problem: Structural instability during ascent/descent phases.")

    return {
        "status": status,
        "diagnostics": feedback
    }

## 5. Statistical Performance Evaluation
To critically verify our model's performance beyond raw accuracy, we extract comprehensive classification metrics from the resulting confusion matrix components: True Positives ($TP$), False Positives ($FP$), True Negatives ($TN$), and False Negatives ($FN$).

### 5.1 Evaluation Metrics
1. **Precision (Positive Predictive Value):** Measures the fidelity of form error diagnoses.
   $$\text{Precision} = \frac{TP}{TP + FP}$$
2. **Recall (Sensitivity):** Reflects the framework's capability to flag faulty form.
   $$\text{Recall} = \frac{TP}{TP + FN}$$
3. **F1-Score:** The harmonic mean of precision and recall, balancing structural diagnostic errors.
   $$\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
4. **Area Under the Receiver Operating Characteristic (ROC-AUC):** Quantifies the probability that the classifier ranks a randomly chosen positive instance higher than a randomly chosen negative one, evaluated across all operational discrimination thresholds:
   $$\text{AUC} = \int_{0}^{1} \text{Sensitivity}(\alpha) \cdot \frac{d}{d\alpha}(1 - \text{Specificity}(\alpha)) \, d\alpha$$

In [ ]:
# A simple verification test cell in your notebook
import joblib
import os

def test_model_persistence():
    assert os.path.exists("../models/squat_classifier_model.pkl"), "Model file missing!"
    assert os.path.exists("../models/squat_scaler.pkl"), "Scaler file missing!"
    
    test_model = joblib.load("../models/squat_classifier_model.pkl")
    print("Smoke Test Passed: Model structure loaded successfully from storage.")

test_model_persistence()

Smoke Test Passed: Model structure loaded successfully from storage.
